In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(threadName)s - %(levelname)s - %(message)s",
)

In [ ]:
from bgm_archive.duck import DuckdbStorage
from tqdm import tqdm
import logging

logger = logging.getLogger(__name__)

DB_PATH = "bgm-graph-try1.duckdb"
# DB_PATH = ':memory:'

bg = DuckdbStorage(db=DB_PATH)

In [ ]:
bg.setup_db()

In [ ]:
from bgm_archive.loader import WikiArchiveLoader

loader = WikiArchiveLoader(archive_path="../dataset/dump-2025-06-03.210251Z.zip")
bg.load_all(loader, progress_bar=True)
# logger.info("inserted %d subjects", inserted_count)

In [ ]:
with bg.open_db() as conn:
    scp_df = conn.execute(
        "SELECT summary, COUNT(1) AS cnt FROM PersonCharacter GROUP BY summary ORDER BY cnt DESC "
    ).df()
scp_df

In [ ]:
with bg.open_db() as conn:
    char_roles_df = conn.execute(
        "SELECT role, COUNT(1) AS cnt FROM Characters GROUP BY role ORDER BY cnt DESC "
    ).df()
char_roles_df

In [ ]:
with bg.open_db() as conn:
    result = conn.execute("""
    SELECT s2, e
    FROM GRAPH_TABLE(
    bgm_graph
                          MATCH (s1: Subjects)-[e: s2s]-(s2: Subjects)
                          COLUMNS (s1, s2, e)
                          )
                          WHERE s1.id = 1
""").fetchall()
result[0][0]

In [ ]:
with bg.open_db(read_only=True) as conn:
    result = conn.execute("""
    SELECT Persons p, Subjects s, Characters c, PersonCharacter.summary
FROM PersonCharacter
INNER JOIN Persons ON PersonCharacter.person_id = Persons.id
INNER JOIN Subjects ON PersonCharacter.subject_id = Subjects.id
INNER JOIN Characters ON PersonCharacter.character_id = Characters.id
LIMIT 10
""").fetchall()
result

In [ ]:
with bg.open_db(read_only=True) as conn:
    result = conn.execute("""
WITH Engagements AS (
    SELECT Persons p, Subjects s, Characters c, PersonCharacter pc
    FROM PersonCharacter
    INNER JOIN Persons ON PersonCharacter.person_id = Persons.id
    INNER JOIN Subjects ON PersonCharacter.subject_id = Subjects.id
    INNER JOIN Characters ON PersonCharacter.character_id = Characters.id
),
S2C AS (
    SELECT Subjects s, Characters c, SubjectCharacter sc
    FROM SubjectCharacter
    INNER JOIN Subjects ON SubjectCharacter.subject_id = Subjects.id
    INNER JOIN Characters ON SubjectCharacter.character_id = Characters.id
),
S2P AS (
    SELECT Subjects s, Persons p, SubjectPersons sp
    FROM SubjectPersons
    INNER JOIN Subjects ON SubjectPersons.subject_id = Subjects.id
    INNER JOIN Persons ON SubjectPersons.person_id = Persons.id
),
S2S AS (
    SELECT s1, s2, SubjectRelation sr
    FROM SubjectRelation
    INNER JOIN Subjects s1 ON SubjectRelation.subject_id = s1.id
    INNER JOIN Subjects s2 ON SubjectRelation.related_subject_id = s2.id
)
    SELECT * FROM S2S LIMIT 10
""").fetchall()
result